# Day 1 Lab — Eval-first SFT on a Small Pretrained LM

**Goal:** use a real, small pretrained causal LM on CPU, run baseline generation on the existing support/incident/tool-use examples, apply basic data gates, then run a tiny SFT update.

This is intentionally minimal and classroom-friendly. The default model is `HuggingFaceTB/SmolLM2-135M-Instruct`; if model download is unavailable, the notebook falls back to `sshleifer/tiny-gpt2`.

In [ ]:
# If needed, uncomment the install line below in a fresh notebook environment.
# %pip install -q "transformers>=4.41" "accelerate>=0.30" torch

import copy, math, random, re, json
from pprint import pprint
import torch
from torch.optim import AdamW
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(7)
torch.manual_seed(7)
DEVICE = "cpu"
# Keep the default very small for CPU. Swap to Qwen/Qwen2.5-0.5B-Instruct if your laptop has enough RAM.
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
FALLBACK_MODEL = "sshleifer/tiny-gpt2"

def load_small_lm(model_name=MODEL_NAME):
    try:
        tokenizer = AutoTokenizer.from_pretrained(model_name)
        model = AutoModelForCausalLM.from_pretrained(model_name)
        loaded = model_name
    except Exception as exc:
        print(f"Could not load {model_name}: {exc}\nFalling back to {FALLBACK_MODEL}.")
        tokenizer = AutoTokenizer.from_pretrained(FALLBACK_MODEL)
        model = AutoModelForCausalLM.from_pretrained(FALLBACK_MODEL)
        loaded = FALLBACK_MODEL
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model.to(DEVICE)
    model.train()
    print("Loaded:", loaded)
    return tokenizer, model

def make_tiny_trainable(model, keywords=("lm_head", "embed_tokens", "wte")):
    """Freeze most parameters so CPU training remains quick."""
    for p in model.parameters():
        p.requires_grad = False
    trainable = []
    for name, p in model.named_parameters():
        if any(k in name for k in keywords):
            p.requires_grad = True
            trainable.append(name)
    if not trainable:
        # Generic fallback: unfreeze the last few tensors.
        for name, p in list(model.named_parameters())[-8:]:
            p.requires_grad = True
            trainable.append(name)
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable tensors: {len(trainable)} | trainable params: {n_params:,}")
    return trainable

def generate_text(model, tokenizer, prompt, max_new_tokens=48):
    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(out[0], skip_special_tokens=True)

def sft_loss(model, tokenizer, prompt, target, max_length=256):
    full = prompt + target
    enc = tokenizer(full, return_tensors="pt", truncation=True, max_length=max_length).to(DEVICE)
    prompt_ids = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=max_length).input_ids
    labels = enc.input_ids.clone()
    # Train only completion tokens; mask prompt tokens.
    prompt_len = min(prompt_ids.shape[1], labels.shape[1])
    labels[:, :prompt_len] = -100
    outputs = model(**enc, labels=labels)
    return outputs.loss

def run_sft_steps(model, tokenizer, train_rows, steps=8, lr=5e-4):
    make_tiny_trainable(model)
    opt = AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    losses = []
    for step in range(steps):
        row = train_rows[step % len(train_rows)]
        loss = sft_loss(model, tokenizer, row["prompt"], row["target"])
        opt.zero_grad()
        loss.backward()
        opt.step()
        losses.append(float(loss.detach().cpu()))
    print("losses:", [round(x, 3) for x in losses])
    return losses

## 1. Existing examples from the lab
These are the same examples already used in the Day 1 lab: refund summary, incident brief, and tool-action selection.

In [ ]:
examples = [
    {"id":"refund-1", "slice":"support",
     "instruction":"Summarize refund status for a support manager.",
     "context":"Refund approved Jun 5. Expected Jun 10. Owner: Billing.",
     "target":"Refund approved Jun 5; expected Jun 10. Owner: Billing."},
    {"id":"incident-1", "slice":"incident",
     "instruction":"Brief this incident for an engineering manager.",
     "context":"Impact: 21 tenants. Cause: token refresh bug. Owner: Auth. Next action: hotfix.",
     "target":"21 tenants are impacted by a token refresh bug. Auth owns the hotfix next action."},
    {"id":"tool-1", "slice":"tool_use",
     "instruction":"State the next tool action for this task.",
     "context":"User asks for recent unread emails. Required source: mailbox search. Constraint: do not search chats.",
     "target":"Search unread emails in the mailbox and summarize only the relevant email results."},
]

def format_prompt(ex):
    return f"Instruction: {ex['instruction']}\nContext: {ex['context']}\nAnswer: "

train_rows = [{"prompt": format_prompt(ex), "target": ex["target"]} for ex in examples]
pprint(train_rows[0])

## 2. Baseline generation before tuning

In [ ]:
tokenizer, model = load_small_lm()
for ex in examples:
    print("\n---", ex["id"], "---")
    print(generate_text(model, tokenizer, format_prompt(ex), max_new_tokens=36))

## 3. Data gates before tuning
The gate is simple and intentionally inspectable: schema validity, target length, and weak grounding overlap with trusted context.

In [ ]:
def quality_gate(ex):
    required = ["instruction", "context", "target", "slice"]
    schema_ok = all(ex.get(k) for k in required)
    target_words = set(re.findall(r"[A-Za-z0-9]+", ex["target"].lower()))
    context_words = set(re.findall(r"[A-Za-z0-9]+", ex["context"].lower()))
    overlap = len(target_words & context_words) / max(1, len(target_words))
    return {"schema_ok": schema_ok, "overlap": round(overlap, 2), "passes": schema_ok and overlap >= 0.35 and len(ex["target"].split()) <= 40}

for ex in examples:
    ex["quality"] = quality_gate(ex)
    print(ex["id"], ex["quality"])

clean = [ex for ex in examples if ex["quality"]["passes"]]
print("clean examples:", [x["id"] for x in clean])

## 4. Tiny CPU SFT update
This is not production training. It tunes a small trainable subset for a few steps so participants can observe real model behavior changes.

In [ ]:
clean_train_rows = [{"prompt": format_prompt(ex), "target": ex["target"]} for ex in clean]
_ = run_sft_steps(model, tokenizer, clean_train_rows, steps=10, lr=8e-4)

## 5. Generate after tuning

In [ ]:
for ex in examples:
    print("\n---", ex["id"], "---")
    print(generate_text(model, tokenizer, format_prompt(ex), max_new_tokens=36))

## Discussion
- Did the model become more likely to use the target answer structure?
- Which data gate is too weak for real SFT?
- Which eval slice should be frozen before scaling training?